# 01 · arXiv — graph + vector in one store

A property graph of papers, authors and categories, with pgvector HNSW over the
same Paper entities. This notebook tours **query, read and mutation over one
store** — Cypher analytics, vector search, graph expansion and GraphRAG, then
the `get`/`get_triplets` read API and a full upsert → get → delete lifecycle.

> Run `prepare.py` first to build the `arxiv` graph (this notebook reads it).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))  # examples/demos
from _common import agens, config
from _common.models import EMBED_DIM, get_embed_model, get_llm, configure_settings
config.require_openai_key()  # reads OPENAI_API_KEY from examples/demos/.env

# open the existing graph (built by prepare.py); vector_dimension lets vector_query use HNSW
store = agens.make_pg_store("arxiv", vector_dimension=EMBED_DIM, create=False)
embed_model, llm = get_embed_model(), get_llm()

## The graph

An element is written on the label naming what it is -- `Paper`, `Author`,
`Category` -- and every one of those inherits `"__Node__"`, so matching the base
label still reaches all of them. Counting by type reads each label's own storage
and nothing else: measured on twenty thousand of each of two types, 248 buffers
against 495 through a btree over a scalar copy of the label, which also cost a
property in every element and an index entry on every write.


In [2]:
import pandas as pd

# One statement, grouped by the label. This asked for a `__type__` scalar that no
# longer exists -- the label is the type.
rows = store.structured_query("""MATCH (n:"__Node__")
    RETURN label(n) AS type, count(*) AS count ORDER BY count DESC""")
edges = store.structured_query(
    'MATCH (:"__Node__")-[r]->(:"__Node__") RETURN count(*) AS c'
)[0]["c"]
print(f"relationships: {edges:,}")
pd.DataFrame(rows)


relationships: 2,655


,type,count
0,Author,1671
1,Paper,600
2,Category,96


## (a) Analytics — plain Cypher

Aggregations walk the **edges** (cheap — the edge implies the endpoint type) and
use `count(*)` (not `count(p)`, which would materialize each paper's embedding).

In [3]:
pd.DataFrame(store.structured_query('''
    MATCH (p:"__Node__")-[:"AUTHORED_BY"]->(a:"__Node__")
    RETURN a.name AS author, count(*) AS papers ORDER BY papers DESC LIMIT 10'''))

,author,papers
0,I. Grabec,6
1,Maxim A. Yurkin,4
2,Alfons G. Hoekstra,4
3,Debashish Goswami,3
4,O. G. Tudose,3
5,S. Schrader,3
6,J. P. Hague,3
7,Valeri P. Maltsev,3
8,M. Prelipceanu,3
9,Ignazio Licata,3


In [4]:
pd.DataFrame(store.structured_query('''
    MATCH (p:"__Node__")-[:"IN_CATEGORY"]->(c:"__Node__")
    RETURN c.name AS category, count(*) AS papers ORDER BY papers DESC LIMIT 10'''))

,category,papers
0,astro-ph,111
1,hep-th,55
2,hep-ph,49
3,quant-ph,42
4,cond-mat.mtrl-sci,41
5,gr-qc,33
6,cond-mat.str-el,33
7,cond-mat.mes-hall,29
8,cond-mat.stat-mech,24
9,nucl-th,21


In [5]:
pd.DataFrame(store.structured_query('''
    MATCH (p:"__Node__") WHERE p.year IS NOT NULL
    RETURN p.year AS year, count(*) AS papers ORDER BY year DESC LIMIT 10'''))

,year,papers
0,2023,2
1,2022,2
2,2021,2
3,2019,5
4,2017,2
5,2016,11
6,2015,24
7,2014,6
8,2013,6
9,2012,11


## (b) Vector search — HNSW over Paper abstracts

`vector_query` embeds the question and runs an HNSW nearest-neighbour search,
returning entities + cosine scores.

In [6]:
from llama_index.core.vector_stores.types import VectorStoreQuery
question = "graph neural networks for molecular property prediction"
qv = embed_model.get_query_embedding(question)
nodes, scores = store.vector_query(VectorStoreQuery(query_embedding=qv, similarity_top_k=5))
for n, s in zip(nodes, scores):
    print(f"{s:.3f}  {(n.properties or {}).get('title', n.name)[:80]}")

0.430  Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from Input Cat
0.370  Genetic Optimization of Photonic Bandgap Structures
0.353  Potfit: effective potentials from ab-initio data
0.350  Simulation of Robustness against Lesions of Cortical Networks
0.341  Weighted percolation on directed networks


## (c) Graph expansion — `get_rel_map`

Expand the vector hits through the graph (their authors and categories).

In [7]:
for src, rel, tgt in store.get_rel_map(nodes[:3], depth=1, limit=15):
    sn = (src.properties or {}).get('title', src.name)
    print(f"({str(sn)[:38]}) -[{rel.label}]-> ({tgt.name})")

(Evolutionary Neural Gas (ENG): A Model) -[AUTHORED_BY]-> (Ignazio Licata)
(Evolutionary Neural Gas (ENG): A Model) -[AUTHORED_BY]-> (Luigi Lella)
(Evolutionary Neural Gas (ENG): A Model) -[IN_CATEGORY]-> (physics.gen-ph)
(Evolutionary Neural Gas (ENG): A Model) -[IN_CATEGORY]-> (q-bio.PE)
(Genetic Optimization of Photonic Bandg) -[AUTHORED_BY]-> (Joel Goh)
(Genetic Optimization of Photonic Bandg) -[AUTHORED_BY]-> (Ilya Fushman)
(Genetic Optimization of Photonic Bandg) -[AUTHORED_BY]-> (Dirk Englund)
(Genetic Optimization of Photonic Bandg) -[AUTHORED_BY]-> (Jelena Vuckovic)
(Genetic Optimization of Photonic Bandg) -[IN_CATEGORY]-> (physics.optics)
(Genetic Optimization of Photonic Bandg) -[IN_CATEGORY]-> (physics.comp-ph)
(Potfit: effective potentials from ab-i) -[AUTHORED_BY]-> (Peter Brommer)
(Potfit: effective potentials from ab-i) -[AUTHORED_BY]-> (Franz Gähler)
(Potfit: effective potentials from ab-i) -[IN_CATEGORY]-> (cond-mat.mtrl-sci)


## (d) GraphRAG

Attach a `PropertyGraphIndex` to the populated store (`kg_extractors=[]` so it
never re-extracts) and answer with a grounded LLM response.

In [8]:
from llama_index.core import PropertyGraphIndex
from llama_index.core.indices.property_graph import VectorContextRetriever
index = PropertyGraphIndex.from_existing(store, embed_model=embed_model, llm=llm,
                                         kg_extractors=[], use_async=False)
qe = index.as_query_engine(sub_retrievers=[
    VectorContextRetriever(graph_store=store, embed_model=embed_model,
                           similarity_top_k=5, path_depth=1, include_text=True)], llm=llm)
print(qe.query(question))

Graph neural networks are increasingly being utilized in the field of molecular property prediction, leveraging their ability to model complex relationships and interactions within molecular structures. These networks can effectively capture the graph-like nature of molecules, where atoms are represented as nodes and bonds as edges, allowing for the extraction of meaningful features that correlate with various molecular properties. This approach has shown promise in enhancing the accuracy of predictions related to chemical properties, biological activities, and material characteristics.


## (e) Read by id & triplets — `get` / `get_triplets`

Fetch specific nodes by id (or property) and the triplets around them — the
read side of the property-graph API, reusing the vector hits from (b) as seeds.

In [9]:
ids = [n.name for n in nodes[:3]]
for n in store.get(ids=ids):                       # fetch specific Paper nodes by id
    print(f"[{n.name}] {(n.properties or {}).get('title', n.name)[:70]}")
print("\ntriplets leaving the top paper:")
for s, r, t in store.get_triplets(entity_names=ids[:1]):
    print(f"  ({s.name}) -[{r.label}]-> ({t.name})")

[0704.0598] Evolutionary Neural Gas (ENG): A Model of Self Organizing Network from
[0704.0185] Potfit: effective potentials from ab-initio data
[0704.0181] Genetic Optimization of Photonic Bandgap Structures

triplets leaving the top paper:
  (0704.0598) -[AUTHORED_BY]-> (Ignazio Licata)
  (0704.0598) -[AUTHORED_BY]-> (Luigi Lella)
  (0704.0598) -[IN_CATEGORY]-> (physics.gen-ph)
  (0704.0598) -[IN_CATEGORY]-> (q-bio.PE)


## (f) Mutation lifecycle — `upsert` / `get` / `delete`

A throwaway `crud_demo` graph (on its own `from_conf` engine) so the populated
`arxiv` graph is never mutated: upsert a few nodes, read them back by property
and via triplets, delete by id and by name, then clear.

In [10]:
from llama_index.core.graph_stores.types import EntityNode, Relation
from llama_index_agensgraph.engine import AgensEngine
from llama_index_agensgraph.graph_stores.agensgraph import AgensPropertyGraphStore

scratch_engine = AgensEngine.from_conf(config.conf())   # from_conf: build a pool from a conf dict
scratch = AgensPropertyGraphStore("crud_demo", conf=config.conf(), vector_dimension=EMBED_DIM,
                                  create=True, refresh_schema=False, engine=scratch_engine)
scratch.structured_query('MATCH (n:"__Node__") DETACH DELETE n')  # reset

scratch.upsert_nodes([
    EntityNode(name="demo:p1", label="Paper", properties={"title": "Graphs for X", "year": 2024}),
    EntityNode(name="demo:p2", label="Paper", properties={"title": "Graphs for Y", "year": 2025}),
    EntityNode(name="demo:alice", label="Author")])
scratch.upsert_relations([
    Relation(label="AUTHORED_BY", source_id="demo:p1", target_id="demo:alice"),
    Relation(label="AUTHORED_BY", source_id="demo:p2", target_id="demo:alice")])

print("get(properties={'year': 2025}):", [n.name for n in scratch.get(properties={"year": 2025})])
print("triplets:", [(s.name, r.label, t.name)
                    for s, r, t in scratch.get_triplets(entity_names=["demo:p1", "demo:p2"])])
scratch.delete(ids=["demo:p1"]); scratch.delete(entity_names=["demo:alice"])
print("remaining:", [n.name for n in scratch.get(ids=["demo:p1", "demo:p2", "demo:alice"])])
scratch.structured_query('MATCH (n:"__Node__") DETACH DELETE n')  # leave it empty
scratch_engine.close()

get(properties={'year': 2025}): ['demo:p2']
triplets: [('demo:p1', 'AUTHORED_BY', 'demo:alice'), ('demo:p2', 'AUTHORED_BY', 'demo:alice')]
remaining: ['demo:p2']


## How it was built

`prepare.py` streams arXiv from Hugging Face and ingests it directly (no LLM
extraction), then embeds the papers in parallel:

```python
store.upsert_nodes([EntityNode(name=paper_id, label="Paper",
                               properties={"title": t, "abstract": a, "year": y}),
                    EntityNode(name=author, label="Author")])
store.upsert_relations([Relation(label="AUTHORED_BY", source_id=paper_id, target_id=author)])
await store.aupsert_nodes([EntityNode(name=paper_id, label="Paper", embedding=vec)])
```

One AgensGraph graph now serves analytics, vector search, graph expansion and
GraphRAG — no separate graph DB and vector DB.

In [11]:
agens.close()